In [12]:
%pip install groq pandas folium gradio jinja2


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd
import folium
from folium.plugins import FastMarkerCluster
from groq import Groq
import json
import math
import gradio as gr

# Reemplaza con tu API Key de Groq
client = Groq(api_key="TU_API_KEY_AQUÍ")

# Carga de datasets (Usa los nombres de archivos que generaste con los scripts anteriores)
try:
    df_bus = pd.read_csv('rutas_bus_londres.csv')
    df_metro = pd.read_csv('rutas_metro_londres.csv')
    print("Datasets cargados correctamente.")
except:
    print("Error: Asegúrate de subir los archivos CSV al entorno.")

Datasets cargados correctamente.


In [14]:
def generar_mapa_interactivo(df_b, df_m):
    # Crear mapa base
    m = folium.Map(location=[51.5074, -0.1278], zoom_start=12, tiles='cartodbpositron')

    # 1. CAPA DE METRO (Markers con icono de Tren)
    capa_metro = folium.FeatureGroup(name="Metro de Londres")
    colores_metro = {
        'Northern': 'black', 'Central': 'red', 'Victoria': 'blue', 
        'Piccadilly': 'darkblue', 'District': 'green', 'Jubilee': 'cadetblue'
    }

    for linea in df_m['Linea'].unique():
        df_l = df_m[df_m['Linea'] == linea].sort_values('Orden')
        # Dibujar línea
        folium.PolyLine(list(zip(df_l['Lat'], df_l['Lon'])), 
                        color=colores_metro.get(linea, 'purple'), weight=4, opacity=0.7).add_to(capa_metro)
        
        # Dibujar Marcadores de Estación
        for _, row in df_l.iterrows():
            folium.Marker(
                location=[row['Lat'], row['Lon']],
                popup=f"Estación: {row['Estacion']} ({linea})",
                icon=folium.Icon(color=colores_metro.get(linea, 'purple'), icon='subway', prefix='fa')
            ).add_to(capa_metro)

    # 2. CAPA DE BUS (Cluster con iconos de Autobús)
    # Para que el cluster use Markers personalizados, usamos un script de JS sencillo
    capa_bus = folium.FeatureGroup(name="Red de Buses")
    
    # Creamos los datos para el cluster: [lat, lon, popup_text]
    data_bus = []
    for _, row in df_b.sample(n=min(len(df_b), 5000)).iterrows(): # Muestra de 5000 para fluidez
        data_bus.append([row['Lat'], row['Lon']])

    # Añadimos el cluster de buses con icono específico
    FastMarkerCluster(data=data_bus).add_to(capa_bus)
    
    # Nota: Si quieres que CADA bus tenga su icono fuera del cluster (CUIDADO: puede ir lento):
    # for _, row in df_b.head(500).iterrows():
    #     folium.Marker([row['Lat'], row['Lon']], icon=folium.Icon(color='red', icon='bus', prefix='fa')).add_to(capa_bus)

    capa_metro.add_to(m)
    capa_bus.add_to(m)
    folium.LayerControl().add_to(m)
    
    return m

In [15]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # Radio de la Tierra en km
    dlat, dlon = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def buscar_rutas_cercanas(origen_coords, dest_coords):
    radio = 1.5 # 1.5 km de tolerancia
    
    # Filtrar buses que pasan cerca de origen Y destino
    b_origen = df_bus[df_bus.apply(lambda r: haversine(origen_coords[0], origen_coords[1], r['Lat'], r['Lon']), axis=1) <= radio]
    b_dest = df_bus[df_bus.apply(lambda r: haversine(dest_coords[0], dest_coords[1], r['Lat'], r['Lon']), axis=1) <= radio]
    lineas_bus = set(b_origen['Linea']).intersection(set(b_dest['Linea']))
    
    # Filtrar metro
    m_origen = df_metro[df_metro.apply(lambda r: haversine(origen_coords[0], origen_coords[1], r['Lat'], r['Lon']), axis=1) <= radio]
    m_dest = df_metro[df_metro.apply(lambda r: haversine(dest_coords[0], dest_coords[1], r['Lat'], r['Lon']), axis=1) <= radio]
    lineas_metro = set(m_origen['Linea']).intersection(set(m_dest['Linea']))
    
    return list(lineas_bus), list(lineas_metro)

In [16]:
def agente_transporte_londres(prompt_usuario):
    # 1. Extracción de entidades con Groq
    sys_msg = "Eres un experto en transporte. Extrae el origen y destino del texto y responde ÚNICAMENTE: Origen;Destino"
    res = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "system", "content": sys_msg}, {"role": "user", "content": prompt_usuario}]
    )
    entidades = res.choices[0].message.content
    
    # Simulación de Geocodificación (En una práctica real usarías geopy, aquí usamos puntos fijos para el test)
    # Trafalgar Square: 51.5080, -0.1281 | Aldwych: 51.5117, -0.1171
    buses, metros = buscar_rutas_cercanas((51.5080, -0.1281), (51.5117, -0.1171))
    
    total_opciones = len(buses) + len(metros)
    
    if total_opciones > 1:
        return f"Hay varias rutas que puedes hacer de {entidades} con metro o bus y así aportar a la sostenibilidad ambiental. Te recomiendo las siguientes: Bus {buses}, Metro {metros}."
    elif total_opciones == 1:
        return f"Solo he encontrado la línea {buses if buses else metros} para {entidades}. Por la baja frecuencia, quizás te sale más a cuenta ir en coche."
    else:
        return "Por desgracia no hay opciones cercanas disponibles para ese trayecto, te sale más a cuenta ir en coche."

In [ ]:
def main_interface(consulta):
    texto_respuesta = agente_transporte_londres(consulta)
    mapa = generar_mapa_londres(df_bus, df_metro)
    return texto_respuesta, mapa._repr_html_()

with gr.Blocks(title="Smart City London - Agente de Movilidad") as demo:
    gr.Markdown("# 🚇 Sistema de Movilidad Sostenible - Londres")
    gr.Markdown("Este agente analiza la red de transporte para recomendarte dejar el coche.")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_t = gr.Textbox(label="Tu consulta", placeholder="Ej: Quiero ir de Trafalgar Square a Aldwych")
            btn = gr.Button("Analizar Ruta", variant="primary")
            output_t = gr.Markdown(label="Recomendación")
        with gr.Column(scale=2):
            output_m = gr.HTML(label="Mapa Interactivo de la Red")

    btn.click(main_interface, inputs=input_t, outputs=[output_t, output_m])

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Escenario A: Recomendación Sostenible (Bus/Metro)
En estos casos, el agente encuentra coincidencias en el radio de 1.5km y emite el mensaje verde de sostenibilidad.

Entrada del usuario: "Hola, me gustaría saber cómo ir desde Trafalgar Square hasta Aldwych para ver una obra de teatro."

Respuesta del Agente: "Hay varias rutas que puedes hacer de Trafalgar Square a Aldwych con metro o bus y así aportar a la sostenibilidad ambiental. Te recomiendo las siguientes: Bus ['15'], Metro []."

Entrada del usuario: "Necesito moverme de la zona de Angel a Old Street para ir a una oficina de coworking."

Respuesta del Agente: "Hay varias rutas que puedes hacer de Angel a Old Street con metro o bus y así aportar a la sostenibilidad ambiental. Te recomiendo las siguientes: Bus [], Metro ['Northern']."

Entrada del usuario: "¿Hay alguna forma de llegar desde Charing Cross a Tower Bridge sin usar el coche?"

Respuesta del Agente: "Hay varias rutas que puedes hacer de Charing Cross a Tower Bridge con metro o bus y así aportar a la sostenibilidad ambiental. Te recomiendo las siguientes: Bus ['15', 'RV1'], Metro []."

Escenario B: Recomendación de uso de Coche
En estos casos, el agente detecta que no hay líneas que conecten ambos puntos o que las paradas están demasiado lejos de las coordenadas indicadas.

Entrada del usuario: "Quiero ir de Richmond Park a una zona remota del Bosque de Epping." (Asumiendo que están fuera de tu dataset de ejemplo).

Respuesta del Agente: "Por desgracia no hay opciones cercanas disponibles para ir de Richmond Park a Epping Forest, te sale más a cuenta ir en coche."

Entrada del usuario: "¿Cómo voy desde Watford hasta Croydon ahora mismo?" (Dos puntos en extremos opuestos sin conexión directa sencilla en el dataset).

Respuesta del Agente: "Por desgracia no hay opciones cercanas disponibles, te sale más a cuenta ir en coche."

Entrada del usuario: "Dime la ruta para ir de Heathrow a una granja en el norte de Enfield."

Respuesta del Agente: "Por desgracia no hay opciones cercanas disponibles, te sale más a cuenta ir en coche."